# Exercise 20.3: LoRA Low-Rank Adaptation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-20/03-lora-low-rank-adaptation.ipynb)

- **Book Reference:** Chapter 20 (*Embeddings, Vector Search (RAG) and Fine-Tuning (LoRA)*, Section 20.2)
- **Estimated Time:** 15–20 minutes
- **Learning Objectives:**
  1. Compute full fine-tuning vs. LoRA parameter counts ($d^2$ vs. $2dr$) and compression ratios.
  2. Implement a LoRA linear layer $W = W_0 + \frac{\alpha}{r} B A$ with frozen $W_0$ and low-rank factors $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$.
  3. Verify empirically that $\operatorname{rank}(B A) \le r$ and reproduce the Chapter 20 worked example ($d=4096, r=8$).

In [ ]:
import numpy as np

## Mathematical Background

**Low-Rank Adaptation (LoRA)** freezes a pre-trained weight matrix $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$ and learns a low-rank update:
$$W = W_0 + \frac{\alpha}{r} B A, \qquad B \in \mathbb{R}^{d_{\text{out}} \times r}, \quad A \in \mathbb{R}^{r \times d_{\text{in}}}, \quad r \ll \min(d_{\text{in}}, d_{\text{out}})$$

### Worked Example Check (Chapter 20.2)
For a $4096 \times 4096$ matrix ($d = 4096$) and rank $r = 8$:
- Full fine-tuning parameters: $4096^2 = 16{,}777{,}216$
- LoRA trainable parameters: $2 \times 4096 \times 8 = 65{,}536$ ($\approx 0.39\%$ of full fine-tuning).

In [ ]:
def lora_parameter_counts(d_in: int, d_out: int, rank: int) -> tuple[int, int, float]:
    """Compute full vs LoRA parameter counts and fraction.

    Args:
        d_in: Input dimension.
        d_out: Output dimension.
        rank: LoRA rank r.

    Returns:
        Tuple (full_params, lora_params, lora_fraction) where
        full_params = d_in * d_out, lora_params = rank * (d_in + d_out),
        and lora_fraction = lora_params / full_params.
    """
    # TODO: Compute and return (full_params, lora_params, lora_fraction)
    raise NotImplementedError("Complete this TODO")


def init_lora_weights(
    d_in: int, d_out: int, rank: int, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Initialise LoRA factors B (d_out, rank) as zeros and A (rank, d_in) as small Gaussian.

    Args:
        d_in: Input dimension.
        d_out: Output dimension.
        rank: LoRA rank r.
        seed: Random seed.

    Returns:
        Tuple (B, A).
    """
    # TODO: Use np.random.default_rng(seed) for A ~ N(0, 0.02^2) and np.zeros for B
    raise NotImplementedError("Complete this TODO")


def merge_lora_weights(
    W0: np.ndarray, B: np.ndarray, A: np.ndarray, alpha: float = 8.0
) -> np.ndarray:
    """Merge frozen W0 with scaled low-rank update (alpha / r) * (B @ A)."""
    # TODO: Infer rank r = A.shape[0] and return W0 + (alpha / r) * (B @ A)
    raise NotImplementedError("Complete this TODO")


def lora_forward(
    x: np.ndarray,
    W0: np.ndarray,
    B: np.ndarray,
    A: np.ndarray,
    alpha: float = 8.0,
) -> np.ndarray:
    """Compute forward pass y = x @ W0.T + (alpha / r) * ((x @ A.T) @ B.T).

    Args:
        x: Input batch of shape (batch, d_in).
        W0: Frozen weight matrix of shape (d_out, d_in).
        B: LoRA matrix of shape (d_out, r).
        A: LoRA matrix of shape (r, d_in).
        alpha: LoRA scaling factor.

    Returns:
        Output batch of shape (batch, d_out).
    """
    # TODO: Compute base output plus scaled low-rank branch without forming d_out x d_in Delta W
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint</summary>

- At initialisation `B = 0`, so `B @ A == 0` and the adapted model starts identical to `W0`.
- In `lora_forward`, computing `(x @ A.T) @ B.T` keeps intermediate activations of width `r` instead of materialising the full `d_out x d_in` update.

</details>

In [ ]:
# 1. Worked example: d=4096, r=8
full_p, lora_p, frac = lora_parameter_counts(4096, 4096, 8)
assert full_p == 16_777_216
assert lora_p == 65_536
assert abs(frac - 65_536 / 16_777_216) < 1e-12

# 2. Exercise 20.3: d=1000, r=4
_, lora_ex, _ = lora_parameter_counts(1000, 1000, 4)
assert lora_ex == 8000

# 3. Forward equivalence and rank bound
rng = np.random.default_rng(42)
d_in, d_out, r = 64, 64, 4
W0 = rng.normal(0.0, 0.1, size=(d_out, d_in))
B, A = init_lora_weights(d_in, d_out, r, seed=42)
x = rng.normal(size=(5, d_in))

# At init (B=0), lora_forward matches W0 exactly
assert np.allclose(lora_forward(x, W0, B, A), x @ W0.T)

# After simulated training update to B
B_trained = rng.normal(0.0, 0.05, size=(d_out, r))
delta_W = B_trained @ A
assert np.linalg.matrix_rank(delta_W) <= r, "Rank of BA must be <= r"

W_merged = merge_lora_weights(W0, B_trained, A, alpha=8.0)
assert np.allclose(lora_forward(x, W0, B_trained, A, alpha=8.0), x @ W_merged.T)
print(f"All checks passed! 4096x4096 r=8 uses {frac:.4%} parameters; rank(BA)={np.linalg.matrix_rank(delta_W)} <= {r}")

## Think Deeper

1. Why is $B$ initialised to all zeros while $A$ is initialised with small Gaussian noise?
2. Why does $\operatorname{rank}(BA) \le r$ hold for any $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$ (Exercise 20.4)?